# EDA — Encuesta de Sueldos IT Argentina (Sysarmy 2026.1)

Exploración inicial del dataset crudo antes de aplicar la limpieza definitiva
en `src/clean.py`.

In [1]:
import pandas as pd

RAW_PATH = "../data/raw/sueldos_2026_1.csv"

# El CSV exportado desde Google Sheets trae 9 filas de encabezado/nota legal
# antes de la fila real de columnas.
df = pd.read_csv(RAW_PATH, skiprows=9)
df.shape

(4939, 60)

In [2]:
df.head()

,donde_estas_trabajando,dedicacion,tipo_de_contrato,ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos,ultimo_salario_mensual_o_retiro_neto_en_pesos_argentinos,pagos_en_dolares,si_tu_sueldo_esta_dolarizado_cual_fue_el_ultimo_valor_del_dolar_que_tomaron,recibis_algun_tipo_de_bono,a_que_esta_atado_el_bono,tuviste_actualizaciones_de_tus_ingresos_laborales_durante_el_ultimo_semestre,...,cuanto_cobras_por_guardia,aclara_el_numero_que_ingresaste_en_el_campo_anterior,salir_o_seguir_contestando_sobre_estudios,tengo_edad,genero,habias_respondido_nuestra_encuesta_en_ediciones_anteriores,cuales_consideras_que_son_las_mejores_empresas_de_it_para_trabajar_en_este_momento_en_tu_ciudad,sueldo_dolarizado,seniority,_sal
0,Mendoza,Full-Time,Staff (planta permanente),3000000.0,2400000.0,Mi sueldo está dolarizado (pero cobro en moned...,1300,No,Performance individual,No,...,NaN,NaN,NaN,45,Mujer Cis,Sí,NaN,True,Senior,3000000.0
1,Santa Fe,Full-Time,Contractor,5000000.0,5000000.0,Cobro todo el salario en dólares,1400,No,No recibo bono,No,...,NaN,NaN,NaN,50,Mujer Cis,Sí,NaN,True,Senior,5000000.0
2,Buenos Aires,Full-Time,Tercerizado (trabajo a través de consultora o ...,4600000.0,3600000.0,Cobro parte del salario en dólares,1436,No,No recibo bono,Uno,...,0.0,Porcentaje de mi sueldo bruto,Terminar encuesta,36,Mujer Cis,Sí,NaN,True,Senior,4600000.0
3,Ciudad Autónoma de Buenos Aires,Full-Time,Staff (planta permanente),7800000.0,6000000.0,Cobro parte del salario en dólares,1470,3+ sueldos,Mix de las anteriores,No,...,NaN,NaN,NaN,60,Hombre Cis,Sí,NaN,True,Senior,7800000.0
4,Jujuy,Full-Time,Contractor,3500000.0,3200000.0,Cobro todo el salario en dólares,NaN,No,No recibo bono,No,...,NaN,NaN,NaN,29,Hombre Cis,Sí,Nose,True,Semi-Senior,3500000.0


## Calidad de datos: nulos y tipos

In [3]:
cols_interes = [
    "donde_estas_trabajando", "seniority", "trabajo_de", "modalidad_de_trabajo",
    "genero", "ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos",
]
df[cols_interes].isna().sum()

donde_estas_trabajando                                       0
seniority                                                    0
trabajo_de                                                   0
modalidad_de_trabajo                                         0
genero                                                       1
ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos    0
dtype: int64

## Distribución del sueldo bruto

In [4]:
salario = df["ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos"]
salario.describe()

count    4.939000e+03
mean     3.876029e+06
std      2.492699e+06
min      2.000000e+05
25%      2.166336e+06
50%      3.268000e+06
75%      4.944960e+06
max      2.000000e+07
Name: ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos, dtype: float64

In [5]:
# Percentiles extremos: usamos p1-p99 como criterio de recorte de outliers
# en la limpieza, en vez de un umbral fijo.
salario.quantile([0.01, 0.05, 0.5, 0.95, 0.99])

0.01      580000.0
0.05     1120950.4
0.50     3268000.0
0.95     8700000.0
0.99    12978340.0
Name: ultimo_salario_mensual_o_retiro_bruto_en_pesos_argentinos, dtype: float64

## Categorías clave

In [6]:
df["seniority"].value_counts()

seniority
Senior         2892
Semi-Senior    1514
Junior          533
Name: count, dtype: int64

In [7]:
df["modalidad_de_trabajo"].value_counts()

modalidad_de_trabajo
100% remoto                      2352
Híbrido (presencial y remoto)    2160
100% presencial                   427
Name: count, dtype: int64

## Conclusiones de la EDA

- El dataset ya viene relativamente limpio (versión "CLEAN" de Openqube): sin
  sueldos negativos o en cero, valores dentro de un rango plausible.
- Se recortan outliers por percentil 1-99 en `src/clean.py` para evitar que
  errores de tipeo (ej. un sueldo cargado con ceros de más) distorsionen las
  medianas.
- `seniority`, `modalidad_de_trabajo` y `genero` tienen categorías limpias,
  listas para agrupar sin normalización adicional.
- El análisis completo y el dashboard interactivo están en
  `app/streamlit_app.py`.